In [8]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic() # --> automatically uses Anthropic api key from env

response = client.messages.create(
    model = 'claude-haiku-4-5',
    max_tokens=100,
    messages=[
        {'role': 'user', 'content': 'Say hello in exactly 5 words.'}
    ]
)

print(response.content[0].text)

Hello, how are you today?


In [1]:
import requests
url = 'http://export.arxiv.org/api/query?search_query=cat:cs.AI+OR+cat:cs.LG+OR+cat:cs.CL&sortBy=submittedDate&sortOrder=descending&max_results=10'
response = requests.get(url)
print(response.text[:2000])

<?xml version='1.0' encoding='UTF-8'?>
<feed xmlns:opensearch="http://a9.com/-/spec/opensearch/1.1/" xmlns:arxiv="http://arxiv.org/schemas/atom" xmlns="http://www.w3.org/2005/Atom">
  <id>https://arxiv.org/api/L0myrzrt2iKcr6+2FZ+ET9w4Qjw</id>
  <title>arXiv Query: search_query=cat:cs.AI OR cat:cs.LG OR cat:cs.CL&amp;id_list=&amp;start=0&amp;max_results=10</title>
  <updated>2026-05-08T15:14:52Z</updated>
  <link href="https://arxiv.org/api/query?search_query=cat:cs.AI+OR+(cat:cs.LG+OR+cat:cs.CL)&amp;start=0&amp;max_results=10&amp;id_list=" type="application/atom+xml"/>
  <opensearch:itemsPerPage>10</opensearch:itemsPerPage>
  <opensearch:totalResults>421948</opensearch:totalResults>
  <opensearch:startIndex>0</opensearch:startIndex>
  <entry>
    <id>http://arxiv.org/abs/2605.06667v1</id>
    <title>ActCam: Zero-Shot Joint Camera and 3D Motion Control for Video Generation</title>
    <updated>2026-05-07T17:59:58Z</updated>
    <link href="https://arxiv.org/abs/2605.06667v1" rel="altern

In [2]:
import feedparser

parsed = feedparser.parse(response.content)
print(len(parsed.entries), "entries")
print(parsed.entries[0].keys())

10 entries
dict_keys(['id', 'guidislink', 'link', 'title', 'title_detail', 'updated', 'updated_parsed', 'links', 'summary', 'summary_detail', 'tags', 'published', 'published_parsed', 'arxiv_comment', 'arxiv_primary_category', 'authors', 'author_detail', 'author'])


In [3]:
for i in range(len(parsed.entries)):
    title = parsed.entries[i]['title']
    summary = parsed.entries[i]['summary']
    link = parsed.entries[i]['link']
    authors = [a['name'] for a in parsed.entries[i]['authors']]
    print(", ".join(authors))
    print(link)
    print(title)
    print(summary, '\n')

Omar El Khalifi, Thomas Rossi, Oscar Fossey, Thibault Fouque, Ulysse Mizrahi, Philip Torr, Ivan Laptev, Fabio Pizzati, Baptiste Bellot-Gurlet
https://arxiv.org/abs/2605.06667v1
ActCam: Zero-Shot Joint Camera and 3D Motion Control for Video Generation
For artistic applications, video generation requires fine-grained control over both performance and cinematography, i.e., the actor's motion and the camera trajectory. We present ActCam, a zero-shot method for video generation that jointly transfers character motion from a driving video into a new scene and enables per-frame control of intrinsic and extrinsic camera parameters. ActCam builds on any pretrained image-to-video diffusion model that accepts conditioning in terms of scene depth and character pose. Given a source video with a moving character and a target camera motion, ActCam generates pose and depth conditions that remain geometrically consistent across frames. We then run a single sampling process with a two-phase conditioning

In [11]:
papers = []
for entry in parsed.entries:
    papers.append({
        "title": entry["title"],
        "abstract": entry["summary"],
        "link": entry["link"],
        "authors": [a["name"] for a in entry["authors"]],
    })

print(len(papers), "papers ready")

10 papers ready


In [14]:
for paper in papers[:5]:
    abstract = paper['abstract']

    response = client.messages.create(
        model='claude-haiku-4-5',
        max_tokens=1000,
        messages=[
            {'role':'user', 'content':f'Summarize the following abstract by specifying Problem, Approach, Key Result and Why it matters in an easy way. Do not include a title or any preamble. Start directly with ## Problem. Use exactly these four headers, identical wording each time: ## Problem, ## Approach, ## Key Result, ## Why It Matters. No bold inside headers. Each section should be 1 to 3 sentences. Total summary under 200 words. Use bullets only when there are 3+ distinct items to list. Otherwise prefer prose. Start summarization right away, do not include anything not related to the abstract. Explain technical terms briefly when they are central to understanding the approach {abstract}'}
        ]
    )

    print(f"## {paper['title']}\n")
    print(response.content[0].text)
    print('\n' + "="*60 + '\n')

## ActCam: Zero-Shot Joint Camera and 3D Motion Control for Video Generation

## Problem

Creating videos with artistic control over both actor motion and camera movement is challenging. Existing methods struggle to simultaneously transfer a character's movements from one video to a new scene while allowing precise control over camera angles and positions for each frame.

## Approach

ActCam uses a pretrained image-to-video diffusion model (a neural network trained to generate video frames) that takes two types of conditioning inputs: the character's pose (body position and orientation) and scene depth (distance information). The method generates geometrically consistent pose and depth conditions frame-by-frame, then applies a two-phase guidance strategy where early generation steps use both pose and depth to establish scene structure, followed by pose-only guidance to refine fine details without over-constraining the output.

## Key Result

ActCam outperforms pose-only methods and oth

In [2]:
import sys
sys.path.insert(0, '..')
from src.email_sender import send_digest_email

test_digest = "This is a test digest to check if the email is being sent"

send_digest_email(test_digest, "Test Digest Checking")

Email sent successfully


True

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

sender = os.getenv("SENDER_EMAIL")
password = os.getenv("EMAIL_APP_PASSWORD")

print(f"Sender: {sender}")
print(f"Password length: {len(password) if password else 'None'}")
print(f"Password repr: {repr(password)}")

Sender: esrefbedel@gmail.com
Password length: 19
Password repr: 'fscl jkpk txrq ibag'


In [13]:
print(repr(os.getenv("SENDER_EMAIL")))
print(repr(os.getenv("RECEPIENT_EMAIL")))

'esrefbedel@gmail.com'
'esrefbedel@gmail.com'


In [1]:
import json
import requests

url = "https://huggingface.co/api/daily_papers?limit=10"

response = requests.get(url)
data = response.json()

print(json.dumps(data[0], indent=2))

{
  "paper": {
    "id": "2605.08513",
    "authors": [
      {
        "_id": "6a033b7e86b054ce2fa40922",
        "name": "Hamid Kazemi",
        "hidden": false
      },
      {
        "_id": "6a033b7e86b054ce2fa40923",
        "name": "Atoosa Chegini",
        "hidden": false
      },
      {
        "_id": "6a033b7e86b054ce2fa40924",
        "name": "Maria Safi",
        "hidden": false
      }
    ],
    "publishedAt": "2026-05-08T00:00:00.000Z",
    "submittedOnDailyAt": "2026-05-12T13:09:55.604Z",
    "title": "A Single Neuron Is Sufficient to Bypass Safety Alignment in Large Language Models",
    "submittedOnDailyBy": {
      "_id": "64f1f9612820a6f1b9e19edd",
      "avatarUrl": "https://cdn-avatars.huggingface.co/v1/production/uploads/64f1f9612820a6f1b9e19edd/hSmhLsczKR1t5jxHwEaE_.jpeg",
      "isPro": false,
      "fullname": "Hamid Kazemi",
      "user": "seyedhamidreza",
      "type": "user",
      "name": "seyedhamidreza"
    },
    "summary": "Safety alignment in languag

In [4]:
import sys
sys.path.insert(0, '..')

In [9]:
from src.fetch_huggingface import fetch_huggingface

hf_papers = fetch_huggingface(limit = 5)
print(f'Got {len(hf_papers)} papers')

for p in hf_papers:
    print(f"[{p['upvotes']} upvotes] {p['title']}")

Got 5 papers
[58 upvotes] Soohak: A Mathematician-Curated Benchmark for Evaluating Research-level Math Capabilities of LLMs
[55 upvotes] Qwen-Image-2.0 Technical Report
[43 upvotes] CollabVR: Collaborative Video Reasoning with Vision-Language and Video Generation Models
[42 upvotes] TMAS: Scaling Test-Time Compute via Multi-Agent Synergy
[27 upvotes] PaperFit: Vision-in-the-Loop Typesetting Optimization for Scientific Documents


In [10]:
hf_papers = fetch_huggingface(limit=20)
for p in hf_papers:
    print(f"[{p['upvotes']} upvotes] {p['title'][:60]}")

[58 upvotes] Soohak: A Mathematician-Curated Benchmark for Evaluating Res
[55 upvotes] Qwen-Image-2.0 Technical Report
[44 upvotes] CollabVR: Collaborative Video Reasoning with Vision-Language
[42 upvotes] TMAS: Scaling Test-Time Compute via Multi-Agent Synergy
[27 upvotes] PaperFit: Vision-in-the-Loop Typesetting Optimization for Sc
[25 upvotes] SEIF: Self-Evolving Reinforcement Learning for Instruction F
[23 upvotes] Geometry Conflict: Explaining and Controlling Forgetting in 
[22 upvotes] WorldReasonBench: Human-Aligned Stress Testing of Video Gene
[22 upvotes] Model Merging Scaling Laws in Large Language Models
[20 upvotes] Auto-Rubric as Reward: From Implicit Preferences to Explicit
[19 upvotes] Memory-Efficient Looped Transformer: Decoupling Compute from
[13 upvotes] Key-Value Means
[12 upvotes] Pixal3D: Pixel-Aligned 3D Generation from Images
[11 upvotes] Dynamic Skill Lifecycle Management for Agentic Reinforcement
[11 upvotes] Rebellious Student: Reversing Teacher Signals for R

In [1]:
def extract_arxiv_id(link):
    try:
        id_part = link.split("/abs/")[-1]
        return id_part.split("v")[0]
    except:
        return None
    
print(extract_arxiv_id("https://arxiv.org/abs/2605.13846v1"))  
print(extract_arxiv_id("https://arxiv.org/abs/2605.08513"))    

2605.13846
2605.08513


In [2]:
arxiv_ids = {"2605.13846"}

hf_papers = [
    {"title": "Duplicate paper", "link": "https://arxiv.org/abs/2605.13846"},
    {"title": "Unique paper", "link": "https://arxiv.org/abs/2605.99999"},
]

In [4]:
filtered = [p for p in hf_papers if extract_arxiv_id(p['link']) not in arxiv_ids]
print(len(filtered))
print(filtered[0]['title'])

1
Unique paper
